<a href="https://colab.research.google.com/github/Drey332/DIMS/blob/main/CapstoneProject/01_PaperReviewSummary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import urllib.request
import os
import pandas as pd

# I am using http://dataminingtutorial.com as the companion site referenced in the paper.
# These are the specific files I need to download as outlined in the methodology.
base_url = "http://dataminingtutorial.com/files/"
files_to_download = ["users.csv", "likes.csv", "users-likes.csv"]

# I am creating a 'data' directory to keep my Colab environment organized.
os.makedirs("data", exist_ok=True)

print("Starting my dataset download...\n")

for file_name in files_to_download:
    url = base_url + file_name
    save_path = os.path.join("data", file_name)

    try:
        print(f"Downloading {file_name}...")
        # Here I download the individual file from the companion site.
        urllib.request.urlretrieve(url, save_path)
        print(f"Successfully saved to {save_path}")
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")

print("\nDownload process complete. I am now verifying the files...\n")

# I will now verify my downloads by loading the first few rows of 'users.csv'.
try:
    users_df = pd.read_csv("data/users.csv", nrows=5)
    print("Preview of my users.csv dataframe:")
    display(users_df.head())
except Exception as e:
    print("I could not load users.csv for preview. The companion site might have moved the files.")

Starting my dataset download...

Error downloading users.csv: HTTP Error 404: Not Found
Error downloading likes.csv: HTTP Error 404: Not Found
Error downloading users-likes.csv: HTTP Error 404: Not Found

Download process complete. I am now verifying the files...

I could not load users.csv for preview. The companion site might have moved the files.


From what I understand of this paper, the authors are trying to bridge a massive gap between social science and computer science. It seems like psychology has always struggled with a replicability crisis because they rely on small, homogenous samples. Big data (like social media footprints) obviously solves that, but social scientists usually don't have the coding skills to analyze it. On the flip side, computer scientists have the technical chops but lack the psychology theory and human subject ethics. To me, this paper is essentially a tutorial trying to teach researchers how to extract actual psychological insights from massive digital footprints.

They used the myPersonality database, specifically looking at about 110,000 Facebook users in the US. The data basically came in three files: psychodemographics (like age and Big Five personality scores), a dictionary of over 1.5 million unique Facebook Likes, and the mapping file connecting users to what they liked.

What I found really interesting was how they handled the sheer size of it. Because one user only likes a tiny fraction of all possible pages, the User-Footprint matrix is mostly empty (less than 0.006% density). They had to use a "sparse matrix format" that only records the non-zero values. That one trick dropped the RAM requirement from 1.4 TB down to just 270 MB, which makes total sense but is a cool workaround.

Honestly, the math in this section was a little hard to understand at first, but here is how I broke down their step-by-step pipeline:

Singular Value Decomposition (SVD): They used this to shrink the massive sparse matrix into a smaller set of dimensions. The tricky part for me was understanding why they didn't center the data first but it turns out doing that would destroy the sparsity, fill the matrix with negative means, and instantly crash the RAM. They ran SVD uncentered and then rotated it later to make the dimensions readable.

Latent Dirichlet Allocation (LDA): I know this is usually for NLP text topics, but they cleverly adapted it to cluster Facebook Likes together based on how often the same users liked them.

Predictive Modeling: Once the data was shrunk down via SVD and LDA, they just used standard linear and logistic regression to predict real-life traits (like using the clusters to predict someone's Openness or Gender).

Cross-Validation: To prove they weren't just overfitting to random noise, they used 10-fold cross-validation on unseen testing data.

4. Challenges & Ethical Limitations
What really stood out to me here were the ethical warnings. They made a great point that just because data can be easily scraped from the web, it doesn't mean users actually consented to having their psychology analyzed. The "network effect" makes this even messier if one person consents, their profile still exposes behavioral data about their friends who definitely didn't consent.
One thing I’m still wondering aboutand what I want to bring up for discussion is how scalable this exact architecture actually is. This paper shows SVD and LDA working on a single machine using sparse formats. But given what we know about distributed computing, I want to ask: how do these specific matrix factorization techniques hold up when you migrate them to an HDFS environment using MapReduce or Spark? Once a dataset outgrows even a single machine's sparse memory limits, dealing with uncentered data across a distributed cluster seems like a completely different beast.

